# 04 — Explore visually

Explore the project's three story angles before building the publication
charts, using the shared Pillow templates. Charts render inline via `display()`.

- **A. Domestic** — adjusted vs nominal gross (dumbbell)
- **B. Worldwide** — domestic vs international split (dumbbell)
- **C. Genre** — which genres skew international vs domestic (diverging bars)

> Note: this project renders with the shared **Pillow** factory rather than
> matplotlib. (matplotlib does not run in this project's Python 3.14 venv — a
> known `MarkerStyle` deepcopy recursion during axis-tick rendering — and the
> publication path is Pillow anyway, so matplotlib isn't a dependency here.)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import pandas as pd, duckdb
from src.ingest import load_config
from chart_templates import lollipop, single_ranked_bars, stacked_100pct_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: this notebook only reads, so it runs even if another notebook
# kernel has the DuckDB file open (DuckDB is single-writer).
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15').df()
films['label'] = films['title'] + '  (' + films['release_year'].astype(str) + ')'
img_w, img_h, _ = PRESETS['twitter_landscape']
def money(v):
    return f'${abs(v)/1e9:.2f}B' if abs(v) >= 1e9 else f'${abs(v)/1e6:.0f}M'
films[['title','adjusted_gross','nominal_gross','release_year']].head()

## A. Domestic — adjusted vs nominal
The gap between what a film made at the time (gold) and its inflation-adjusted
gross (teal). Domestic (U.S. & Canada) only.

In [ ]:
display(lollipop(
    films, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='All-time top 15 domestic films, adjusted for inflation',
    subtitle='U.S. & Canada gross in constant 2026 dollars (CPI-U). Gold dot = what each film made at the time (nominal).',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Today\u2019s $', value2_label='Nominal', legend_reverse=True,
    img_width=img_w, img_height=img_h,
))

## B. Worldwide — domestic vs international
The reason the domestic chart is only half the story: for most top films the
international (rest-of-world) take dwarfs the domestic one. Gold = domestic,
teal = international. Watch **Ne Zha 2** — a Chinese blockbuster with a huge
international total and almost no domestic gross.

A **100% stacked bar** is the right chart for a *split*: each film is one
bar, gold = home %, teal = abroad %. Sorted by share abroad, the teal segment
shrinks steadily down the list — the visual matches the sort. (A dumbbell of
absolute dollars can't do this: its line length is the dollar gap, which
doesn't track the percentage order.)

In [ ]:
# US-made films; top 15 by worldwide gross, ordered by share earned abroad.
ww = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE),
    top AS (SELECT w.title, w.release_year,
                   100.0*w.foreign_gross/w.worldwide_gross AS foreign_pct,
                   100.0*w.domestic_gross/w.worldwide_gross AS home_pct
            FROM films_worldwide w JOIN us ON us.title=w.title AND us.release_year=w.release_year
            ORDER BY w.worldwide_gross DESC LIMIT 15)
    SELECT * FROM top ORDER BY foreign_pct DESC''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
display(stacked_100pct_bars(ww, group_col='label',
    segments=[{'col':'foreign_pct','label':'Rest of world','color':'#005F73'},
              {'col':'home_pct','label':'Home (US/Canada)','color':'#EE9B00'}],
    title='Hollywood films: share of box office earned at home vs abroad',
    subtitle='Top 15 US-made films by worldwide gross, ordered by share earned abroad.',
    bar_height=34, bar_gap=12, img_width=img_w, img_height=img_h))

## C. Share of box office earned abroad, by genre
Plain international share per genre (US-made films): of everything the genre
earned worldwide, what fraction came from outside the U.S. & Canada. Sorted
high to low. Each film's gross is attributed to all its genres (so grosses
double-count across genres — fine for a share, see SOURCES.md). The story:
every genre earns most of its money abroad; sci-fi is simply the lowest.

In [ ]:
genre = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE)
    SELECT gl.genre AS category, COUNT(*) n,
      ROUND(100.0*SUM(w.foreign_gross)/(SUM(w.domestic_gross)+SUM(w.foreign_gross)),1) AS value
    FROM films_worldwide w
    JOIN us ON us.title=w.title AND us.release_year=w.release_year
    JOIN film_genres_long gl ON gl.title=w.title AND gl.release_year=w.release_year
    GROUP BY gl.genre HAVING COUNT(*)>=10 ORDER BY value DESC''').df()
genre['pct_label'] = genre['value'].apply(lambda v: f'{v:.0f}%')
display(single_ranked_bars(genre, category_col='category', value_col='value',
    total_label_col='pct_label', bar_color='#005F73',
    title='Every blockbuster genre earns most of its money abroad',
    subtitle='Share of worldwide box office earned outside the U.S. & Canada, by genre (top US-made films).',
    img_width=img_w, img_height=img_h))

## D. Top foreign-LANGUAGE films by U.S. box office
Non-English-language films ranked by U.S. & Canada lifetime gross (Box Office
Mojo's Foreign Language chart, `films_foreign_us`). Note this is a **language**
grouping: it includes a few U.S.-produced non-English films and, crucially,
*misses English-language films made outside the U.S.* (British, Australian).
Section E below answers the origin-based question and is the honest companion
to this one — keep both to see how language vs. country of origin differ.

Nominal (year-of-release) dollars; country of origin under each title.

In [ ]:
# Top foreign-language films by U.S. (domestic) gross, in today's dollars. Top 15.
foreign = con.execute('''
    SELECT rank_foreign, title, domestic_gross, release_year, distributor, origin_name
    FROM films_foreign_us
    ORDER BY domestic_gross DESC LIMIT 15''').df()
foreign['label'] = foreign['title'] + '  (' + foreign['release_year'].astype(str) + ')'
foreign['gross_label'] = foreign['domestic_gross'].apply(money)
display(foreign[['title','release_year','origin_name','domestic_gross']])
display(single_ranked_bars(foreign, category_col='label', value_col='domestic_gross',
    total_label_col='gross_label', bar_color='#005F73', sublabel_col='origin_name',
    title='Top foreign-language films by U.S. box office',
    subtitle='Non-English-language films by U.S. & Canada lifetime gross, in constant 2026 dollars (CPI-U). Country of origin under each title. Source: Box Office Mojo + BLS.',
    img_width=img_w, img_height=img_h))

## E. Top films CREATED outside the U.S., by U.S. box office
The origin-based cut: of the all-time domestic top 1000, the films whose
**primary country of origin is not the U.S.**, ranked by U.S. & Canada gross
(`films_domestic_all`, `is_foreign`). This is the complete, fact-based answer
to “which films made outside the U.S. did well here?” — and it looks very
different from the language chart: the British/Australian English-language
films (Harry Potter, James Bond, Mad Max, Crocodile Dundee) dominate, none of
which appear in section D.

Caveats stated on the chart: **nominal $** (older titles understated), origin
is TMDB's **primary** country so U.K./U.S. co-productions count as U.K. (a
documented judgment call), and a 2026 title still in theaters can top the list
with an incomplete total (snapshot as of Sep 2026).

In [ ]:
# Non-US-primary-origin films from the domestic top-1000, by U.S. gross (today's $). Top 15.
origin = con.execute('''
    SELECT rank_domestic, title, us_gross, release_year, origin_name
    FROM films_domestic_all
    WHERE is_foreign
    ORDER BY us_gross DESC LIMIT 15''').df()
origin['label'] = origin['title'] + '  (' + origin['release_year'].astype(str) + ')'
origin['gross_label'] = origin['us_gross'].apply(money)
display(origin[['title','release_year','origin_name','us_gross']])
display(single_ranked_bars(origin, category_col='label', value_col='us_gross',
    total_label_col='gross_label', bar_color='#005F73', sublabel_col='origin_name',
    title='Top films created outside the U.S., by U.S. box office',
    subtitle='Domestic top 1000 filtered to non-U.S. primary origin, by U.S. & Canada gross in constant 2026 dollars (CPI-U). Country under each title. Source: Box Office Mojo + TMDB + BLS.',
    img_width=img_w, img_height=img_h))

## F. Top films made outside the Hollywood studio system, by U.S. box office
The third lens on “foreign,” and the closest to “not a Hollywood film.”
Section E used country of *origin*, which follows production-company
registration and filming location — so it labels Nolan's *The Odyssey* and the
Harry Potter / Bond films British, even though U.S. studios (Universal, Warner,
MGM) financed and released them. This cut instead flags a film as
**non-Hollywood only when NO U.S.-registered studio is among its production
companies** (`is_non_hollywood`).

The result is a much shorter, stricter list (~13 of the top 1000): the
genuinely independent-of-Hollywood hits — Crocodile Dundee (Rimfire, AU),
Slumdog Millionaire (Film4, UK), Demon Slayer (ufotable, JP), Lucy (EuropaCorp,
FR), and the EON-only Bond entries.

**This is a documented proxy, not truth.** TMDB's company data is uneven (some
Bond films list MGM, some don't — hence the inconsistency), and a U.S. major
co-financing a British film is a real gray area. Exploration only; the language
cut (D) is the clean, well-defined basis for a published chart.

In [ ]:
# Films with NO US studio among producers (non-Hollywood proxy), by US gross (today's $).
nonhw = con.execute('''
    SELECT rank_domestic, title, us_gross, release_year, origin_name, companies_str
    FROM films_domestic_all
    WHERE is_non_hollywood
    ORDER BY us_gross DESC LIMIT 15''').df()
nonhw['label'] = nonhw['title'] + '  (' + nonhw['release_year'].astype(str) + ')'
nonhw['gross_label'] = nonhw['us_gross'].apply(money)
display(nonhw[['title','release_year','origin_name','us_gross','companies_str']])
display(single_ranked_bars(nonhw, category_col='label', value_col='us_gross',
    total_label_col='gross_label', bar_color='#005F73', sublabel_col='origin_name',
    title='Top films made outside the Hollywood studio system, by U.S. box office',
    subtitle='Domestic top 1000 with no U.S.-registered studio among producers (a documented proxy), by U.S. gross in constant 2026 dollars (CPI-U). Source: Box Office Mojo + TMDB + BLS.',
    img_width=img_w, img_height=img_h))

## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode,
other notebooks).

In [ ]:
con.close()
print('connection closed')

---
**Next:** `06-viz-social.ipynb` builds the publication versions of these three
charts (full titling/source) and saves them to `outputs/social/`.